# TP 1 — Premiers pas avec Python
## Séance 1 : Introduction à la détection d’anomalies et de fraude
### Version corrigée (enseignant)

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Vous travaillez pour une banque. On vous fournit un historique de **5 000 transactions** par carte. Quelques-unes sont frauduleuses. Vous allez :

1. simuler et explorer ces données ;
2. mesurer l’accuracy d’un modèle naïf ;
3. tester une règle simple de détection à seuil ;
4. mesurer la **précision** et le **rappel** de cette règle ;
5. observer comment le seuil change les résultats.

**Rappels**

- Précision = TP / (TP + FP) : parmi les alertes, quelle part est une vraie fraude ?
- Rappel = TP / (TP + FN) : parmi les fraudes réelles, quelle part est détectée ?

## Étape 1 — Génération des données

Exécutez la cellule suivante sans la modifier. Elle crée un DataFrame `df` avec trois colonnes :

- `montant` : montant de la transaction en euros ;
- `heure` : heure de la journée (0 à 23) ;
- `fraude` : 1 si la transaction est frauduleuse, 0 sinon.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 5000

# Transactions normales : montants log-normaux, activité surtout en journée
poids = np.array([1] * 6 + [4] * 16 + [2] * 2, dtype=float)
poids /= poids.sum()

df = pd.DataFrame({
    'montant': rng.lognormal(mean=3.5, sigma=0.6, size=n),
    'heure': rng.choice(24, size=n, p=poids),
    'fraude': 0,
})

# Injection de 25 fraudes : montants plus élevés, plutôt la nuit
idx = rng.choice(n, size=25, replace=False)
df.loc[idx, 'montant'] = rng.lognormal(mean=5.0, sigma=0.6, size=25)
df.loc[idx, 'heure'] = rng.integers(0, 6, size=25)
df.loc[idx, 'fraude'] = 1

df.head()

## Étape 2 — Exploration

Complétez les cellules ci-dessous (remplacez chaque `...`).

In [ ]:
# 1. Nombre de transactions normales (0) et frauduleuses (1)
print(df['fraude'].value_counts())

# 2. Part de transactions frauduleuses (entre 0 et 1)
part_fraude = df['fraude'].mean()
print(f'Part de fraudes : {part_fraude:.2%}')

# 3. Statistiques du montant et de l'heure selon la classe
print(df.groupby('fraude')[['montant', 'heure']].agg(['mean', 'median', 'min', 'max']).round(1))

# 4. Accuracy d'un modèle qui répond toujours « pas de fraude »
accuracy_naif = (df['fraude'] == 0).mean()
print(f'Accuracy du modèle naïf : {accuracy_naif:.2%}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['heure'], df['montant'], c=df['fraude'], cmap='coolwarm', s=12, alpha=0.6)
plt.yscale('log')
plt.xlabel('Heure')
plt.ylabel('Montant (€)')
plt.show()

## Étape 3 — Une règle simple et son évaluation

On définit une règle de détection : **une alerte est levée si le montant dépasse un seuil**.

Complétez la fonction `evaluer` pour qu’elle calcule la matrice de confusion, la précision et le rappel.

In [ ]:
def evaluer(df, alerte):
    """alerte : série de 0/1 (1 = alerte levée). Affiche TP, FP, FN, TN, précision et rappel."""
    tp = ((alerte == 1) & (df['fraude'] == 1)).sum()   # alertes sur des vraies fraudes
    fp = ((alerte == 1) & (df['fraude'] == 0)).sum()   # alertes sur des transactions normales
    fn = ((alerte == 0) & (df['fraude'] == 1)).sum()   # fraudes sans alerte
    tn = ((alerte == 0) & (df['fraude'] == 0)).sum()   # transactions normales sans alerte
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn)
    print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
    print(f'Précision={precision:.2f}  Rappel={rappel:.2f}')
    return precision, rappel

# Règle : alerte si montant > 150 €
alerte = (df['montant'] > 150).astype(int)
evaluer(df, alerte)

## Étape 4 — Influence du seuil

Testez plusieurs seuils et comparez les résultats.

In [ ]:
for seuil in [100, 150, 250, 400]:
    print(f'--- Seuil = {seuil} € ---')
    evaluer(df, (df['montant'] > seuil).astype(int))

## Étape 5 — Ajouter le contexte

Une transaction de 100 € est banale en pleine journée, mais plus suspecte à 3 h du matin. Testez la règle :

**alerte si le montant dépasse 100 € ET si l’heure est avant 6 h.**

In [ ]:
alerte2 = ((df['montant'] > 100) & (df['heure'] < 6)).astype(int)
evaluer(df, alerte2)

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Quelle est la part de transactions frauduleuses ? Que vaut l’accuracy du modèle qui répond toujours « pas de fraude » ? Que peut-on en conclure ?

**Réponse 1.** Il y a 25 fraudes sur 5 000 transactions, soit **0,5 %**. Le modèle qui répond toujours « pas de fraude » obtient une accuracy de **99,5 %**, mais il ne détecte **aucune** fraude (rappel = 0). L’accuracy est donc trompeuse quand les classes sont très déséquilibrées : il faut utiliser la précision et le rappel.

**Question 2.** Observez le nuage de points. Où se situent les fraudes par rapport aux transactions normales ? Les distingue-t-on parfaitement ?

**Réponse 2.** Les fraudes ont des montants nettement plus élevés (médiane d’environ 172 € contre 33 € pour les transactions normales) et ont lieu la nuit (entre 0 h et 5 h). Mais la séparation n’est **pas parfaite** : certaines fraudes ont un montant faible (minimum d’environ 55 €), et il existe des transactions normales avec un montant élevé ou réalisées la nuit. Les deux populations se chevauchent.

**Question 3.** Interprétez la précision et le rappel obtenus avec la règle « montant supérieur à 150 € ».

**Réponse 3.** Avec le seuil 150 € : TP = 16, FP = 31, FN = 9, TN = 4 944. La **précision** vaut environ 0,34 : seulement un tiers des alertes sont de vraies fraudes. Le **rappel** vaut 0,64 : la règle détecte environ 64 % des fraudes, mais en manque 9 sur 25.

**Question 4.** Faites varier le seuil (100, 150, 250, 400). Comment évoluent la précision et le rappel ? Que constate-t-on ?

**Réponse 4.**

| Seuil | TP | FP | FN | Précision | Rappel |
|---|---|---|---|---|---|
| 100 | 21 | 158 | 4 | 0,12 | 0,84 |
| 150 | 16 | 31 | 9 | 0,34 | 0,64 |
| 250 | 5 | 1 | 20 | 0,83 | 0,20 |
| 400 | 3 | 0 | 22 | 1,00 | 0,12 |

Quand le seuil augmente, la **précision augmente** (moins de fausses alertes) mais le **rappel diminue** (plus de fraudes manquées). Il n’existe pas de seuil parfait : choisir un seuil, c’est arbitrer entre les deux.

**Question 5.** Comparez la règle de l’étape 5 avec les règles à seuil de l’étape 4. De quel type d’anomalie s’agit-il (ponctuelle, contextuelle ou collective) ?

**Réponse 5.** La règle contextuelle (montant > 100 ET heure < 6) donne TP = 21, FP = 12, FN = 4, soit une précision de 0,64 et un rappel de 0,84. Avec le même rappel que le seuil 100 € seul (0,84), la précision passe de 0,12 à 0,64 : on a beaucoup moins de fausses alertes. C’est une **anomalie contextuelle** : un montant de 100 € est banal en journée, mais suspect la nuit. Le contexte (ici l’heure) apporte de l’information que le montant seul ne contient pas.

**Question 6.** Pour une banque, vaut-il mieux un seuil bas ou un seuil élevé ? Quels éléments faut-il prendre en compte pour décider ?

**Réponse 6.** Il n’y a pas de réponse unique, cela dépend du **coût d’une fraude manquée** par rapport au **coût d’une fausse alerte**. Éléments à prendre en compte : le montant moyen des fraudes ; le coût d’un client honnête bloqué (insatisfaction, appels au service client) ; la **capacité de l’équipe d’analystes** à traiter les alertes ; les obligations réglementaires ; l’image de la banque. Si les fraudes sont coûteuses et que les analystes peuvent traiter beaucoup d’alertes, on choisit un seuil plutôt bas (rappel élevé). Si les analystes sont débordés ou si bloquer un client coûte cher, on choisit un seuil plus élevé (précision élevée).

## Pour aller plus loin (facultatif)

1. Testez la règle « montant > 75 ET heure < 6 » puis « montant > 150 ET heure < 6 ». Que remarquez-vous ?
2. Modifiez le taux de fraude dans la génération des données (par exemple 100 fraudes au lieu de 25). Comment évoluent la précision et le rappel d’une même règle ?
3. Tracez la précision et le rappel en fonction du seuil sur un même graphique.

## Pistes pour « Pour aller plus loin »

- « montant > 75 ET heure < 6 » : TP = 23, FP = 28, précision 0,45, rappel 0,92. « montant > 150 ET heure < 6 » : TP = 16, FP = 2, précision 0,89, rappel 0,64. Même arbitrage précision / rappel, mais à un meilleur niveau qu’avec le montant seul.
- Avec davantage de fraudes (classes moins déséquilibrées), la précision d’une même règle augmente mécaniquement, car les alertes tombent plus souvent sur de vraies fraudes. La précision dépend donc du taux de fraude (avec 100 fraudes et le seuil 150 €, on obtient environ 0,61 contre 0,34), alors que le rappel ne dépend pas du taux de fraude : il ne varie que par fluctuation d’échantillonnage.
- Pour le graphique : calculer la précision et le rappel dans une boucle sur une liste de seuils, puis tracer les deux courbes avec `plt.plot`. C’est le principe de la courbe précision-rappel vue en séance 10.